<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula03a%20-%20valida%C3%A7%C3%A3o%20cruzada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Aula 03a: Validação Cruzada (Cross-Validation) e Pipelines

Seja muito bem-vindo(a) à aula sobre **Metodologia Científica e Rigor em Machine Learning**!

---

### 🎯 Qual é o objetivo desta aula?
Nos cadernos anteriores, fazíamos uma única divisão entre Treino (80%) e Teste (20%).  
Mas pense comigo:  
* E se demos **sorte** no sorteio e as amostras do teste eram fáceis demais?
* E se demos **azar** e as amostras mais difíceis caíram todas no teste?
* Como podemos confiar que nosso modelo é realmente bom e não apenas sortudo?

Nesta aula, o professor Boldt nos ensina a técnica definitiva para responder a essa pergunta: a **Validação Cruzada (Cross-Validation)** e como construir **Pipelines** para evitar o maior pecado de um cientista de dados: o **Vazamento de Dados (Data Leakage)**!

---

### 🧭 A Trilha Pedagógica do Professor Boldt
1. **O Efeito Sorte/Azar**: Executar o `train_test_split` várias vezes sem semente fixa e ver a acurácia oscilar violentamente entre $91\%$ e $100\%$.
2. **Múltiplos Holdouts**: Rodar 50 divisões aleatórias e tirar a média para diminuir a incerteza.
3. **Validação Cruzada K-Fold artesanal (feita com NumPy)**:
   * Dividir os dados em $K$ partes (pedaços/folds). Cada pedaço tem sua vez de ser o teste enquanto os outros $K-1$ treinam o modelo.
4. **Leave-One-Out (Deixar Um de Fora)**: O caso extremo em que o teste é uma única amostra e treinamos em todo o restante.
5. **O Perigo Invisível do Vazamento de Dados (Data Leakage)**:
   * Por que normalizar o dataset inteiro antes de dividir contamina o teste e gera resultados falsamente otimistas.
6. **Construindo um `CustomPipeline` do zero**: Entendendo a anatomia de um Pipeline do Scikit-Learn.
7. **As Ferramentas Profissionais**: `cross_validate`, `make_pipeline`, `KFold(shuffle=True)`, `LeaveOneOut` e `RepeatedKFold`.



## 1. Ponto de Partida: k-NN com Padronização e Split Fixo (random_state=42)

> Aqui o professor treina dois modelos k-NN ($k=5$ e $k=15$) com uma semente fixa.  
> Com essa semente específica, ambos parecem dar resultados semelhantes (~97.22%). Mas será que isso é sempre verdade?


In [2]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Carregamos os dados
X, y = load_wine(return_X_y=True)

# Divisão inicial com semente fixa (random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Padronizamos com StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo com k=5 vizinhos
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Acurácia k=5 (semente 42): {accuracy_score(y_test, y_pred) * 100:.2f}%")


accuracy: 0.9444444444444444


In [ ]:
# Modelo com k=15 vizinhos com a mesma semente
model = KNeighborsClassifier(n_neighbors=15)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Acurácia k=15 (semente 42): {accuracy_score(y_test, y_pred) * 100:.2f}%")


accuracy: 0.9722222222222222


## 2. A Ilusão do Sorteio Único: Removendo o `random_state`

> **Experimente rodar a célula abaixo várias vezes!**  
> Ao remover a semente (`random_state`), cada execução faz um sorteio diferente.  
> Repare como os valores mudam a cada clique: às vezes $k=5$ ganha, às vezes $k=15$ ganha, às vezes dá 91%, às vezes dá 100%!  
> **Conclusão**: Um único teste não é confiável para tomar decisões sérias.

In [9]:
# Divisão puramente aleatória a cada execução (sem random_state)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Testamos k=5
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Acurácia k= 5 neste sorteio: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# Testamos k=15
model = KNeighborsClassifier(n_neighbors=15)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Acurácia k=15 neste sorteio: {accuracy_score(y_test, y_pred) * 100:.2f}%")


accuracy k= 5: 1.0
accuracy k=15: 0.9166666666666666


## 3. Primeira Solução: Múltiplos Holdouts (Repetir 50 vezes e tirar a média)

> O professor automatiza 50 sorteios independentes e calcula a média final de cada modelo.  
> Isso já reduz bastante o ruído, mas tem um problema: algumas amostras podem ser sorteadas para teste 10 vezes, enquanto outras podem nunca ser testadas!

In [31]:
from pprint import pprint

# Lista de modelos que desejamos comparar
models = [
    ("K= 5", KNeighborsClassifier(n_neighbors=5)),
    ("K=15", KNeighborsClassifier(n_neighbors=15))
]

# Função auxiliar para treinar e avaliar um único modelo
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Realiza um sorteio, escala os dados e avalia todos os modelos
def evaluate_models(models, X, y, scores={}):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    for model_name, model in models:
        acc = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test)
        if model_name not in scores:
            scores[model_name] = []
        scores[model_name].append(acc)
    return scores

# Repete o processo 50 vezes
def multiple_holdouts(models, X, y, n_repeats=50):
    scores = {}
    for _ in range(n_repeats):
        scores = evaluate_models(models, X, y, scores)
    return scores

scores = multiple_holdouts(models, X, y)
print("Média de acurácia após 50 divisões aleatórias:")
for model_name, score_list in scores.items():
    print(f"{model_name}: {sum(score_list) / len(score_list) * 100:.2f}%")


K= 5: 0.9622222222222221
K=15: 0.9638888888888889


## 4. A Solução Científica Perfeita: Validação Cruzada K-Fold (Feita do Zero)

> **Como funciona o K-Fold?**  
> 1. Dividimos o dataset em $K$ pedaços iguais (dobras / folds), por exemplo $K=5$.  
> 2. Rodada 1: Pedaço 1 é teste, pedaços 2, 3, 4, 5 são treino.  
> 3. Rodada 2: Pedaço 2 é teste, pedaços 1, 3, 4, 5 são treino...  
> 4. Repetimos até que **todos os dados** tenham sido testados exatamente uma vez!  

> [!IMPORTANT]  
> **Atenção ao detalhe crucial no código do professor**: O `scaler.fit(X_train)` acontece **dentro do loop de cada fold**! O scaler só aprende a média do treino daquela rodada específica.

In [42]:
import numpy as np

# Validação Cruzada de K Folds feita à mão com NumPy
def cross_validation(models, X, y, n_folds=5, scores={}):
    idxs = np.random.permutation(len(y))
    test_size = len(y) // n_folds
    for i in range(n_folds):
        # 1. Pega os índices deste pedaço para teste
        test_idxs = idxs[i*test_size:(i+1)*test_size]
        X_test, y_test = X[test_idxs], y[test_idxs]
        # 2. Junta todos os outros pedaços para compor o treino
        train_idxs = np.concatenate((idxs[:i*test_size], idxs[(i+1)*test_size:]))
        X_train, y_train = X[train_idxs], y[train_idxs]
        # 3. Ajusta o scaler ESTRITAMENTE nos dados de treino deste fold
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        # 4. Avalia os modelos neste fold
        for model_name, model in models:
            acc = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test)
            if model_name not in scores:
                scores[model_name] = []
            scores[model_name].append(acc)
    return scores

scores = cross_validation(models, X, y)
print("Média do K-Fold (5 folds) feito do zero:")
for model_name, score_list in scores.items():
    print(f"{model_name}: {sum(score_list) / len(score_list) * 100:.2f}%")


K= 5: 0.9542857142857143
K=15: 0.9542857142857143


### O Caso Extremo: Leave-One-Out (LOO)
No Leave-One-Out, o número de folds é igual ao número total de linhas ($K = N$). Para cada rodada, deixamos exatamente **1 amostra** para testar e treinamos em todas as outras $N-1$!

In [45]:
# Leave-One-Out: K é igual ao tamanho de y
def leave_one_out(model, X, y, scores={}):
  k = len(y)
  return cross_validation(model, X, y, k, scores)

scores = leave_one_out(models, X, y)
print("Média do Leave-One-Out:")
for model_name, score_list in scores.items():
    print(f"{model_name}: {sum(score_list) / len(score_list) * 100:.2f}%")


K= 5: 0.9719101123595506
K=15: 0.9662921348314607


## 5. Usando o `cross_validate` do Scikit-Learn

> O scikit-learn tem a função oficial `cross_validate`.  
> Mas cuidado! Veja o que acontece se passarmos `X` puro sem normalizar: o k-NN sem escala tem um desempenho medíocre (~69%).  
> E se normalizarmos `X` antes de chamar o `cross_validate`? Cometeríamos **Data Leakage**!

In [50]:
from sklearn.model_selection import cross_validate

model = KNeighborsClassifier()
# Executa validação cruzada padrão (5 folds por padrão)
scores = cross_validate(model, X, y)
pprint(scores)


{'fit_time': array([0.00809193, 0.00671673, 0.0067687 , 0.00581884, 0.0058434 ]),
 'score_time': array([0.02274275, 0.01404929, 0.01505136, 0.01493382, 0.0077455 ]),
 'test_score': array([0.72222222, 0.66666667, 0.63888889, 0.65714286, 0.77142857])}


In [51]:
# Média dos 5 folds sem escala (resultado baixo: ~69%)
print("Acurácia média sem escala:", scores['test_score'].mean())


0.6912698412698413


## 6. A Grande Solução para o Vazamento de Dados: Construindo um Pipeline do Zero!

> **O que é um Pipeline?**  
> É uma esteira de produção que amarra o pré-processamento (como o `StandardScaler`) junto com o modelo preditivo (`KNeighborsClassifier`).  
> * Quando chamamos `.fit(X_train, y_train)`: a esteira calcula a média no treino e treina o modelo.  
> * Quando chamamos `.predict(X_test)`: a esteira usa **a mesma média do treino** para transformar o teste e faz a predição!  

O professor implementa a classe `CustomPipeline` do zero para mostrar a elegância desse mecanismo!

In [67]:
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin

# Implementação didática de um Pipeline sequencial
class CustomPipeline(BaseEstimator, ClassifierMixin, TransformerMixin):
    def __init__(self, steps):
      self.steps = steps
    
    def fit(self, X, y):
      T = X
      for i in range(len(self.steps)):
        # Se a etapa for um transformador (como StandardScaler), aplica fit_transform
        if hasattr(self.steps[i], "fit_transform"):
          T = self.steps[i].fit_transform(T, y)
        else:
          # Se for o classificador final, apenas ajusta (fit)
          T = self.steps[i].fit(T, y)
      return self
    
    def predict(self, X):
      T = X
      for i in range(len(self.steps)):
        # Na hora de prever, apenas transforma com as médias aprendidas no treino
        if hasattr(self.steps[i], "transform"):
          T = self.steps[i].transform(T)
        else:
          T = self.steps[i].predict(T)
      return T

# Montamos nossa esteira de processamento protegida contra vazamento de dados
pipe = CustomPipeline([
    StandardScaler(),
    KNeighborsClassifier()
])

# Teste em um split simples
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print("Acurácia com CustomPipeline:", accuracy_score(y_test, y_pred))


accuracy 0.9444444444444444


## 7. Passando o Pipeline para o `cross_validate` e a Pegadinha do `KFold` sem Shuffle

> Agora podemos passar o `pipe` direto para o `cross_validate`! O `cross_validate` vai rodar o scaler internamente em cada fold sem nenhum vazamento!  
> Mas olhe a célula 13: o array `y` original está ordenado por classes (todos os 0 primeiro, depois os 1, depois os 2).  
> Se usarmos `KFold(n_splits=5)` sem embaralhar (`shuffle=False`), um fold inteiro pode conter apenas uma classe e o modelo falha feio!  
> **Solução Obrigatória**: Sempre use `KFold(n_splits=5, shuffle=True)`!

In [74]:
# Avaliando o Pipeline dentro do cross_validate
scores = cross_validate(pipe, X, y)
print("Acurácia média com Pipeline:", scores['test_score'].mean())


0.8934920634920636


In [76]:
scores = cross_validate(pipe, X, y, cv=5)
print("Acurácia média com cv=5:", scores['test_score'].mean())


0.8934920634920636


In [78]:
from sklearn.model_selection import KFold

# KFold SEM embaralhamento (shuffle padrão é False!)
spliter = KFold(n_splits=5)
scores = cross_validate(pipe, X, y, cv=spliter)
print("Acurácia com KFold sem shuffle (cuidado!):", scores['test_score'].mean())


0.8934920634920636


In [80]:
# Veja como o y vem perfeitamente ordenado: [0, 0, ..., 1, 1, ..., 2, 2]
y


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2])

In [96]:
# KFold COM embaralhamento (shuffle=True) - agora sim!
spliter = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipe, X, y, cv=spliter)
print(f"Acurácia correta com shuffle=True: {scores['test_score'].mean() * 100:.2f}%")


0.9661904761904763


## 8. Ferramentas Profissionais Prontas do Scikit-Learn

* `make_pipeline(StandardScaler(), KNeighborsClassifier())`: cria o pipeline oficial em uma única linha.
* `LeaveOneOut()`: objeto divisor para leave-one-out.
* `RepeatedKFold(n_splits=5, n_repeats=10)`: repete o 5-Fold 10 vezes (total de 50 testes) para máxima robustez estatística!

In [104]:
from sklearn.pipeline import make_pipeline

# make_pipeline cria o pipeline de forma elegante e concisa
pipe = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)
spliter = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(pipe, X, y, cv=spliter)
print(f"Acurácia via make_pipeline oficial: {scores['test_score'].mean() * 100:.2f}%")


0.9550793650793651


In [118]:
from sklearn.model_selection import LeaveOneOut

spliter = LeaveOneOut()
scores = cross_validate(pipe, X, y, cv=spliter)
print(f"Acurácia LeaveOneOut oficial: {scores['test_score'].mean() * 100:.2f}%")


0.9719101123595506
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0.
 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [119]:
from sklearn.model_selection import RepeatedKFold

# 5 Folds repetidos 10 vezes com embaralhamentos diferentes
spliter = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)
scores = cross_validate(pipe, X, y, cv=spliter)
print(f"Acurácia RepeatedKFold (50 folds totais): {scores['test_score'].mean() * 100:.2f}%")


0.964015873015873
[0.94444444 0.97222222 1.         1.         0.91428571 0.94444444
 0.94444444 0.94444444 0.97142857 1.         0.94444444 0.94444444
 0.97222222 0.97142857 1.         1.         0.97222222 0.97222222
 1.         0.88571429 1.         0.91666667 0.97222222 0.91428571
 0.97142857 1.         0.94444444 0.94444444 0.97142857 1.
 0.97222222 0.94444444 0.94444444 0.91428571 0.97142857 0.97222222
 1.         0.91666667 0.91428571 1.         0.97222222 0.97222222
 1.         0.94285714 1.         1.         1.         0.94444444
 0.91428571 0.97142857]


---
## 🎓 Resumo da Aula 03a:
1. **Fim do Sorteio Único**: Um único split treino/teste varia por puro acaso. Avaliar modelos exige repetição e média estatística.
2. **K-Fold Cross-Validation**: Todo exemplo é usado para treino e para teste, garantindo avaliação honesta.
3. **Vazamento de Dados (Data Leakage)**: O erro silencioso mais grave de ML. Transformações (escalas, imputações) devem ser calculadas exclusivamente no treino.
4. **Pipelines**: O encapsulador que impede que dados de teste vazem durante a validação cruzada.
5. **A Importância do `shuffle=True`**: Datasets ordenados por classe exigem embaralhamento antes de fatiar os folds.

👉 **Próximo Passo**: Na **Aula 03b**, aprenderemos como encontrar a melhor combinação de hiperparâmetros de forma 100% automatizada com **GridSearchCV** e **Validação Cruzada Aninhada**!

